# Capítulo 8 — Dados desbalanceados: edição executável

**Importante:** os dados reais do SINAN-TB não foram fornecidos. Este notebook usa **dados sintéticos** para verificar o fluxo computacional; métricas obtidas não reproduzem os resultados do livro. Códigos originais revisados estão em células separadas; preparação, calibração e testes são complementos nossos.

## Preparação reprodutível (complemento)

Geração de dados tabulares sintéticos com desbalanceamento e separação estratificada treino/validação/teste. Nenhuma amostra do teste participa do treinamento, SMOTE ou calibração.

In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from collections import Counter

X, y = make_classification(n_samples=900, n_features=12, n_informative=7,
    n_redundant=2, weights=[0.88, 0.12], flip_y=0.01, random_state=42)
X = pd.DataFrame(X, columns=[f"atributo_{i}" for i in range(X.shape[1])])
y = pd.Series(y, name="obito_sintetico")
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=42)
print("Treino:", X_train.shape, Counter(y_train))
print("Validação:", X_val.shape, Counter(y_val))
print("Teste:", X_test.shape, Counter(y_test))
assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)

Treino: (540, 12) Counter({0: 472, 1: 68})
Validação: (180, 12) Counter({0: 158, 1: 22})
Teste: (180, 12) Counter({0: 158, 1: 22})


### Bloco de Código 8.1 — Aplicação de reamostragem sintética (SMOTE) à base

*Página 120 do PDF · Revisão: confirmado*

**Fonte:** código revisado do livro; conteúdo abaixo preservado.

In [2]:
from collections import Counter
from imblearn.over_sampling import SMOTE

# 1. FUNCAO DE VISUALIZACAO

# Recebe o vetor de rotulos (y) e exibe a contagem de cada classe.
def visualizar_classes(y, titulo):
    counts = Counter(y)
    df_counts = pd.DataFrame.from_dict(counts, orient='index').reset_index()
    df_counts.columns = ['Classe', 'Quantidade']
    print(f'n--- Distribuicao de Classes: {titulo},')
    print(df_counts.to_string(index=False))

# 2. VERIFICACAO DO DESBALANCEAMENTO

visualizar_classes(y_train, "y_train (Original)")

# 3. APLICACAO DO SMOTE

# IMPORTANTE: o SMOTE deve ser aplicado SOMENTE sobre X_train e y_train.
# X_val e X_test permanecem intocados para garantir avaliacao sem vazamento.
# O X_train neste ponto ja passou por imputacao e OHE (apenas variaveis
# numericas e dummies), pois o SMOTE exige entradas completamente numericas.
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Exibe a contagem após o SMOTE: a classe minoritaria (obito=1) e
# sinteticamente expandida até igual a classe majoritaria (obito=0).
visualizar_classes(y_train_res, "y_train (Resampled)")


n--- Distribuicao de Classes: y_train (Original),
 Classe  Quantidade
      0         472
      1          68
n--- Distribuicao de Classes: y_train (Resampled),
 Classe  Quantidade
      0         472
      1         472


In [3]:
assert len(X_train_res) > len(X_train)
assert len(set(Counter(y_train_res).values())) == 1
assert len(X_test) == 180
print("SMOTE: somente treino, classes balanceadas")

SMOTE: somente treino, classes balanceadas


### Bloco de Código 8.2 — Saída do console (output), base SINAN-TB

*Página 120 do PDF · Revisão: confirmado*

**Fonte:** código revisado do livro; conteúdo abaixo preservado.

**Saída ilustrativa do livro (não executável):** o bloco transcrito contém somente o comentário `# Sem código`. Os resultados numéricos do exemplo real não são reproduzidos com a base sintética.

In [4]:
# Sem código


### Bloco de Código 8.3 — Modelo 0, Baseline sem correção do desbalanceamento

*Página 122 do PDF · Revisão: confirmado*

**Fonte:** código revisado do livro; conteúdo abaixo preservado.

In [5]:
xgb_default = xgb.XGBClassifier(random_state=42, eval_metric='logloss')
xgb_default.fit(X_train, y_train)
y_pred_default = xgb_default.predict(X_test)


In [6]:
from sklearn.metrics import classification_report
print("Baseline — teste sintético:")
print(classification_report(y_test, y_pred_default, zero_division=0))

Baseline — teste sintético:
              precision    recall  f1-score   support

           0       0.92      0.99      0.96       158
           1       0.90      0.41      0.56        22

    accuracy                           0.92       180
   macro avg       0.91      0.70      0.76       180
weighted avg       0.92      0.92      0.91       180



### Bloco de Código 8.4 — Saída: Modelo 0, Default

*Página 122 do PDF · Revisão: confirmado*

**Fonte:** código revisado do livro; conteúdo abaixo preservado.

**Saída ilustrativa do livro (não executável):** o bloco transcrito contém somente o comentário `# Sem código`. Os resultados numéricos do exemplo real não são reproduzidos com a base sintética.

In [7]:
# SEM CODIGO


### Bloco de Código 8.5 — Modelo 1, Aprendizado sensível a custo

*Página 123 do PDF · Revisão: confirmado*

**Fonte:** código revisado do livro; conteúdo abaixo preservado.

In [8]:
# Razão negativos/positivos no conjunto de treino
peso_balanceamento = np.sum(y_train == 0) / np.sum(y_train == 1)

xgb_cost = xgb.XGBClassifier(
    scale_pos_weight=peso_balanceamento,
    random_state=42,
    eval_metric='logloss'
)

xgb_cost.fit(X_train, y_train)
y_pred_cost = xgb_cost.predict(X_test)


In [9]:
assert peso_balanceamento > 1
print("Modelo sensível a custo — teste sintético:")
print(classification_report(y_test, y_pred_cost, zero_division=0))

Modelo sensível a custo — teste sintético:
              precision    recall  f1-score   support

           0       0.93      0.99      0.96       158
           1       0.85      0.50      0.63        22

    accuracy                           0.93       180
   macro avg       0.89      0.74      0.79       180
weighted avg       0.92      0.93      0.92       180



### Bloco de Código 8.6 — Saída: Modelo 1, Sensível a custo

*Página 124 do PDF · Revisão: confirmado*

**Fonte:** código revisado do livro; conteúdo abaixo preservado.

**Saída ilustrativa do livro (não executável):** o bloco transcrito contém somente o comentário `# Sem código`. Os resultados numéricos do exemplo real não são reproduzidos com a base sintética.

In [10]:
# Sem código


### Bloco de Código 8.7 — Modelo 3, Balanced Random Forest

*Página 125 do PDF · Revisão: confirmado*

**Fonte:** código revisado do livro; conteúdo abaixo preservado.

In [11]:
from imblearn.ensemble import BalancedRandomForestClassifier

# sampling_strategy='all': balanceia internamente a cada arvore
brf = BalancedRandomForestClassifier(
    sampling_strategy='all',
    replacement=True,
    random_state=42
)

brf.fit(X_train, y_train)
y_pred_brf = brf.predict(X_test)


In [12]:
print("Balanced RF — teste sintético:")
print(classification_report(y_test, y_pred_brf, zero_division=0))

Balanced RF — teste sintético:
              precision    recall  f1-score   support

           0       0.95      0.91      0.93       158
           1       0.50      0.64      0.56        22

    accuracy                           0.88       180
   macro avg       0.72      0.77      0.74       180
weighted avg       0.89      0.88      0.88       180



### Bloco de Código 8.8 — Saída: Modelo 3, Balanced Random Forest

*Página 125 do PDF · Revisão: confirmado*

**Fonte:** código revisado do livro; conteúdo abaixo preservado.

**Saída ilustrativa do livro (não executável):** o bloco transcrito contém somente o comentário `# Sem código`. Os resultados numéricos do exemplo real não são reproduzidos com a base sintética.

In [13]:
# Sem código


### Bloco de Código 8.9 — Varredura de limiares sobre probabilidades calibradas

*Página 127 do PDF · Revisão: confirmado*

**Fonte:** código revisado do livro; conteúdo abaixo preservado.

**Complemento:** calibração Platt usando exclusivamente o conjunto de validação. Modelo de base treinado no conjunto de treino. As probabilidades do teste não são usadas para ajustar o calibrador.

In [14]:
modelo_calibracao = xgb.XGBClassifier(n_estimators=70, max_depth=3, n_jobs=2, random_state=42, eval_metric="logloss")
modelo_calibracao.fit(X_train, y_train)
prob_val = modelo_calibracao.predict_proba(X_val)[:, 1]
platt = LogisticRegression(random_state=42)
platt.fit(prob_val.reshape(-1, 1), y_val)
prob_platt = platt.predict_proba(modelo_calibracao.predict_proba(X_test)[:, 1].reshape(-1, 1))[:, 1]
assert len(prob_platt) == len(y_test)
assert np.isfinite(prob_platt).all()
print("Calibração independente concluída; AUC teste:", round(roc_auc_score(y_test, prob_platt), 3))

Calibração independente concluída; AUC teste: 0.928


In [15]:
from sklearn.metrics import precision_score, recall_score, f1_score

print(f"{'Limiar':>8} {'Precision':>10} {'Recall':>8} {'F1':>6}")
for thr in [0.50, 0.30, 0.18, 0.10]:
    y_pred = (prob_platt >= thr).astype(int)
    p = precision_score(y_test, y_pred, zero_division=0)
    r = recall_score(y_test, y_pred)
    f = f1_score(y_test, y_pred)
    print(f'{thr:>8.2f} {p:>10.2f} {r:>8.2f} {f:>6.2f}')


  Limiar  Precision   Recall     F1
    0.50       0.88     0.32   0.47
    0.30       0.92     0.50   0.65
    0.18       0.81     0.59   0.68
    0.10       0.60     0.68   0.64


### Bloco de Código 8.10 — Saída: tradeoff precision/recall por limiar, base SINAN-TB

*Página 128 do PDF · Revisão: confirmado*

**Fonte:** código revisado do livro; conteúdo abaixo preservado.

In [16]:
from sklearn.metrics import precision_score, recall_score, f1_score

print(f"{'LIMIAR':>8} {'PRECISÃO':>10} {'RECALL':>8} {'F1':>6}")

for thr in [0.50, 0.30, 0.18, 0.10]:
    y_pred = (prob_platt >= thr).astype(int)
    p = precision_score(y_test, y_pred, zero_division=0)
    r = recall_score(y_test, y_pred)
    f = f1_score(y_test, y_pred)
    print(f"{thr:>8.2f} {p:>10.2f} {r:>8.2f} {f:>6.2f}")


  LIMIAR   PRECISÃO   RECALL     F1
    0.50       0.88     0.32   0.47
    0.30       0.92     0.50   0.65
    0.18       0.81     0.59   0.68
    0.10       0.60     0.68   0.64


### Bloco de Código 8.11 — Relatório de classificação: limiar padrão vs. limiar ajustado

*Página 129 do PDF · Revisão: confirmado*

**Fonte:** código revisado do livro; conteúdo abaixo preservado.

In [17]:
from sklearn.metrics import classification_report

print("--- Limiar padrão: 0.50,")

print(classification_report(y_test, (prob_platt >= 0.50).astype(int)))

print("--- Limiar ajustado: 0.18,")

print(classification_report(y_test, (prob_platt >= 0.18).astype(int)))


--- Limiar padrão: 0.50,
              precision    recall  f1-score   support

           0       0.91      0.99      0.95       158
           1       0.88      0.32      0.47        22

    accuracy                           0.91       180
   macro avg       0.89      0.66      0.71       180
weighted avg       0.91      0.91      0.89       180

--- Limiar ajustado: 0.18,
              precision    recall  f1-score   support

           0       0.95      0.98      0.96       158
           1       0.81      0.59      0.68        22

    accuracy                           0.93       180
   macro avg       0.88      0.79      0.82       180
weighted avg       0.93      0.93      0.93       180



### Bloco de Código 8.12 — Saída: comparação dos relatórios de classificação

*Página 130 do PDF · Revisão: confirmado*

**Fonte:** código revisado do livro; conteúdo abaixo preservado.

**Saída ilustrativa do livro (não executável):** o bloco transcrito contém somente o comentário `# Sem código`. Os resultados numéricos do exemplo real não são reproduzidos com a base sintética.

In [18]:
# Sem código
